# Regime Structure 02 — does the up-displacement → shock signal pay for long 0DTE options?

Pre-registration: `discussions/Regime_Structure/005_claude_rs02_preregistration.md`. Code: `src/data_engineering/modeling/regime_c3_options.py`.

- **Signal:** a 15-minute window labelled DISP with an upward net move (RS 01 v1.2 definitions and development-window thresholds), Binance USD-M 1m, BTC/ETH/SOL/XRP. Signal time = window end.
- **Part A (Binance, 2 years):** normalised absolute move $z=\lvert\Delta\log P\rvert/(\hat\sigma_5\sqrt{h/5})$ over 60 minutes and to the next 08:00 UTC settlement; signal vs hour-matched other window ends. Ratio with 97.5% day-bootstrap CI, test window from 2026-04-01.
- **Part B (collector quotes):** long ATM 0DTE straddle at the ask + taker fee, held to settlement (VRP 02 code, approved), entered at the first 5-minute snapshot after a signal. Primary: $M_{\text{signal}}$, mean over sessions of session-mean $R_{\text{net}}$, 95% session bootstrap, VRP verdict labels. Secondary: signal minus all entries; ATM call only; exit at the bid after 60 minutes.

Environment: `RS02_ASSETS` (default BTC,ETH,SOL,XRP), `RS02_OFFLINE=1` (use cached delivery prices), `RS02_SYNTHETIC=1` (smoke run without data, outputs to a temp dir).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import regime_structure as rs, vrp_long_straddle as ls, regime_c3_options as rc
rs, ls, rc = importlib.reload(rs), importlib.reload(ls), importlib.reload(rc)

SYNTHETIC = os.getenv('RS02_SYNTHETIC', '').strip() == '1'
OFFLINE = os.getenv('RS02_OFFLINE', '').strip() == '1'
ASSETS = [a.strip().upper() for a in os.getenv('RS02_ASSETS', ','.join(rc.ASSETS)).split(',') if a.strip()]
P = rc.Params()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='rs02_')); DATA_ROOT = BIN_DIR = None
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    BIN_DIR = Path(os.getenv('BINANCE_UM_DIR') or DATA_ROOT.parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Regime_Structure' / '02'
OUT.mkdir(parents=True, exist_ok=True)
REG = rc.registration(P)
print('RS 02 |', rc.VERSION, '|', rs.VERSION, '| VRP', ls.VERSION, '| synthetic' if SYNTHETIC else '', '| out', OUT)

## 1. Signals on Binance and Part A (does the signal raise the move?)

In [ ]:
SF, ROWS = {}, []
for i, a in enumerate(ASSETS):
    t0 = time.time()
    if SYNTHETIC:
        df = rs.synthetic_klines(n_days=200, seed=30 + i, regime=True, start='2025-12-01')
    else:
        path = BIN_DIR / f'{a}USDT_um_1m.parquet'
        if not path.exists():
            print(f'{a}: no Binance file -> skipped'); continue
        df = pd.read_parquet(path, columns=['open_time', 'close', 'volume', 'taker_buy_volume'])
    SF[a], logc, t0m = rc.signal_frame(df, P)
    ROWS.append(rc.part_a_rows(SF[a], logc, t0m, P).assign(asset=a))
    del df, logc
    print(f'{a}: {SF[a].signal.sum():,} signals of {len(SF[a]):,} windows, {time.time() - t0:.0f}s')
ROWS = pd.concat(ROWS, ignore_index=True)
rng = np.random.default_rng(P.seed)
pa = []
for w in ('dev', 'test'):
    for col in ('z60', 'zsettle'):
        pa.append({'window': w, 'horizon': col, **rc.matched_ratio(ROWS[ROWS.window == w], col, P, rng)})
part_a = pd.DataFrame(pa)
print('Part A — mean normalised |move| after a signal / hour-matched other times (97.5% CI):'); display(part_a.round(4))
pa_asset = pd.DataFrame([{'asset': a, 'horizon': col, **rc.matched_ratio(ROWS[(ROWS.window == 'test') & (ROWS.asset == a)], col,
                         rc.Params(n_boot=1000), rng)} for a in SF for col in ('z60', 'zsettle')])
print('Per asset, test window:'); display(pa_asset[['asset', 'horizon', 'n_signal', 'ratio', 'lo', 'hi']].round(4))

## 2. Part B — executable long options on the collector quotes

In [ ]:
if SYNTHETIC:
    chains, settles = zip(*[rc.synthetic_chain(n_sessions=25, seed=40 + i, under=a) for i, a in enumerate(ASSETS)])
    chain = pd.concat(chains, ignore_index=True); approx = pd.concat(settles, ignore_index=True)
    legs = rc.prepare_legs_grid(chain, 60)
else:
    import duckdb
    DB = DATA_ROOT / 'database' / 'research.duckdb'
    con = duckdb.connect(str(DB), read_only=True)
    try:
        have = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
        cols = [c for c in ls.CHAIN_COLUMNS if c != 'settlement_currency'] + (['settlement_currency'] if 'settlement_currency' in have else [])
        chain = con.execute(f'''
            WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
            SELECT {', '.join('n.' + c for c in cols)}
            FROM normalized_option_chain n
            JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
            WHERE n.minutes_to_expiry BETWEEN {ls.MIN_TTE_MIN - P.exit_after_min} AND {ls.MAX_TTE_MIN}   -- exit legs need TTE down to 60 min
              AND EXTRACT(minute FROM n.state_time) % {P.entry_grid_min} = 0
              AND n.state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}'
              AND abs(ln(n.strike / n.option_forward_price)) < 0.05
              AND n.underlying IN ({','.join(repr(a) for a in SF)})
        ''').df()
        idx = con.execute(f'''SELECT state_time, underlying, futures_index_price FROM time_series_feature_state
                              WHERE state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}' ''').df()
    finally:
        con.close()
    if 'settlement_currency' not in chain:
        chain['settlement_currency'] = None
    legs = rc.prepare_legs_grid(chain, P.entry_grid_min)
st = rc.entry_straddles(legs)
last_signal_time = min(sf['t_end'].max() for sf in SF.values())
st = st[st['state_time'] <= last_signal_time]                      # only entries the Binance signal history covers
sessions = sorted(st.session.unique())
if not SYNTHETIC:
    approx = ls.approx_settlement(idx, sessions)
    names = sorted({ls.index_name(u, c) for u, c in zip(st.underlying, st.settlement_currency)})
    cache = OUT / '02_delivery_prices.csv'
    delivery = (pd.read_csv(cache, dtype={'session': str}) if (OFFLINE and cache.exists())
                else ls.fetch_delivery_prices(names, cache_csv=cache))
    delivery = delivery[delivery.session.isin(sessions)]
else:
    delivery = pd.DataFrame(columns=['index_name', 'session', 'delivery_price'])
x = ls.straddle_pnl(ls.attach_settlement(st, delivery, approx))
signals = pd.concat([sf.loc[sf.signal, ['t_end']].assign(asset=a) for a, sf in SF.items()], ignore_index=True)
if SYNTHETIC:   # synthetic chains are hourly and in another period: place signals on random entry times
    pick = x.drop_duplicates(['state_time', 'underlying']).sample(frac=0.15, random_state=1)
    signals = pd.DataFrame({'asset': pick['underlying'], 't_end': pick['state_time']})
x = rc.tag_signals(x, signals, P)
x = rc.add_variants(x, legs, P)
cov = (x.groupby('underlying').agg(sessions=('session', 'nunique'), entries=('R_net', 'size'),
                                   signal_entries=('signal_entry', 'sum'), first=('state_time', 'min'), last=('state_time', 'max')))
print('settlement source:', x.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict())
display(cov)

In [ ]:
B = rc.part_b(x, P)
print('PRIMARY — signal straddles held to settlement:'); print(json.dumps(B['primary'], indent=2, default=float))
print('VERDICT:', B['primary']['verdict'])
display(B['table'][['metric', 'set', 'n_sessions', 'M', 'ci_low', 'ci_high', 'share_sessions_positive']].round(4))

In [ ]:
sm_sig = ls.session_means(x[x.signal_entry], 'R_net'); sm_all = ls.session_means(x, 'R_net').reindex(sm_sig.index)
fig, ax = plt.subplots(figsize=(10, 3.2))
k = np.arange(len(sm_sig))
ax.bar(k - 0.2, sm_all.values, width=0.4, color='#9aa0a6', label='all entries')
ax.bar(k + 0.2, sm_sig.values, width=0.4, color='#c0392b', label='signal entries')
ax.axhline(0, color='k', lw=0.8); ax.set_xticks(k, sm_sig.index, rotation=90, fontsize=7)
ax.set_ylabel('session mean R_net'); ax.legend(); ax.set_title('Long ATM straddle at the ask, held to settlement')
plt.tight_layout(); plt.show()

## 3. Save compact outputs

In [ ]:
part_a.to_csv(OUT / '02_part_a.csv', index=False)
pa_asset.to_csv(OUT / '02_part_a_by_asset.csv', index=False)
B['table'].to_csv(OUT / '02_part_b_table.csv', index=False)
cov.to_csv(OUT / '02_coverage.csv')
(x.groupby(['session', 'underlying', 'signal_entry'])
  .agg(n=('R_net', 'size'), R_net=('R_net', 'mean'), R_call_net=('R_call_net', 'mean'), R_exit60_net=('R_exit60_net', 'mean'))
  .reset_index().to_csv(OUT / '02_session_asset.csv', index=False, float_format='%.6g'))
manifest = {'registration': REG, 'synthetic': SYNTHETIC, 'assets': list(SF), 'part_a': part_a.to_dict(orient='records'),
            'part_b_primary': B['primary'], 'sessions': [sessions[0], sessions[-1]] if sessions else [],
            'run_utc': str(pd.Timestamp.now(tz='UTC'))}
(OUT / '02_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print('saved to', OUT, sorted(p.name for p in OUT.iterdir()))